# 自定义中间件-Node-style hooks

## 1、基本用法

### 1.1 基于装饰器的实现

In [1]:

from langchain.agents.middleware import before_model, after_model, before_agent, after_agent, AgentMiddleware
from typing import Any
from langgraph.runtime import Runtime
from langchain.agents import AgentState


@before_model
def before_model_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> before_model <-----"
    return None

@after_model
def after_model_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> after_model <-----"
    return None

@before_agent
def before_agent_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> before_agent <-----"
    return None

@after_agent
def after_agent_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> after_agent <-----"
    return None


In [2]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)


agent = create_agent(
    model=model,
    middleware=[
        before_model_middleware,
        after_model_middleware,
        before_agent_middleware,
        after_agent_middleware,
    ]
)


response = agent.invoke({
    "messages": [HumanMessage("你好")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好----> before_agent <---------> before_model <-----
================================== Ai Message ==================================

你好！我在这儿。  
你这条消息里包含了些测试标记（像 `before_agent`、`before_model`），如果你是在做流程/链路调试，我也可以配合。  
想让我帮你做什么？----> after_model <---------> after_agent <-----


### 1.2 基于类的实现

In [3]:
from langchain.agents.middleware import AgentMiddleware

class MyMiddleware(AgentMiddleware):
    def before_model(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> before_model <-----"
        return None

    def after_model(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> after_model <-----"
        return None

    def before_agent(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> before_agent <-----"
        return None

    def after_agent(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> after_agent <-----"
        return None


In [4]:
my_middleware = MyMiddleware()

agent = create_agent(
    model=model,
    middleware=[my_middleware]
)


response = agent.invoke({
    "messages": [HumanMessage("你好")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好----> before_agent <---------> before_model <-----
================================== Ai Message ==================================

你好！🙂  
看起来你在测试消息流/标记格式。我已经接收到内容了。

如果你愿意，我可以继续帮你：
- 解释这段标记含义
- 处理/解析类似格式的文本
- 回答其他问题

你可以直接发下一条。----> after_model <---------> after_agent <-----
